In [35]:
import os
import sys
from dotenv import load_dotenv
import requests
from semanticscholar import AsyncSemanticScholar
import nest_asyncio

nest_asyncio.apply()

load_dotenv(override=True)


# 1. GETTING USER API KEY. DEFAULTS AND CHECKS IP IF KEY IS NOT FOUND
api_key = os.getenv("SEMANTIC_SCHOLAR_API_KEY")
is_key_good = False

print("Searching for User's API Key...")
if api_key is None or api_key == "your_key_here" or api_key.strip() == "":
    print(f"User provided this: '{api_key}' as an API Key. Defaulting to lower limit access...")
    print("Checking IP status...")
    #CHECKING IF IP IS BLOCKED OR NOT
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    query_params = {"query": "test", "limit": 1}
    print("Pinging Semantic Scholar...")
    response = requests.get(url, params=query_params)

    if response.status_code == 200:
        print("Your IP is clear to access Semantic Scholar. Ready to start!")
    elif response.status_code == 429:
        sys.exit("Your IP is currently blocked from using Semantic Scholar.")
else:
    print("API Key found! Going to Semantic Scholar...")
    is_key_good = True


# 2. CONNECTING TO SEMANTIC SCHOLAR
sch = None
if is_key_good:
    sch = AsyncSemanticScholar(api_key=api_key)
else:
    sch = AsyncSemanticScholar()


# 3. VALIDATING THE API KEY
try:
    testing_api_key_works = await sch.get_paper("8ceb75144ecb846bf463e7565e6a18998ae29d1a", fields=["title"])
    # The id for Protein measurement with the Folin phenol reagent. That is the most cited paper on Semantic Scholar.
    print("Your API Key has been approved! Beginning Search...")

except Exception as e:
    print(f"API Key Error: {e}")
    sys.exit("Your API key was not authorized for usage by Semantic Scholar.")


# 4. GETTING TOP 5 RESULTS
results = await sch.search_paper("explanatory gap", limit=5, fields=["title", "year", "abstract"])

print(f"\nSuccess! Found {results.total} total papers.")

if results.total > 0:
    i = 0
    for p in results.items:
        print()
        print(f"Title: {p.title}")
        print(f"Year: {p.year}")
        abstract = getattr(p, "abstract", "No abstract found.")
        print(f"Abstract: {abstract[:100]}..." if abstract else "Abstract: No abstract found.")

        i += 1
        if i > 5:
            break


Searching for User's API Key...
API Key found! Going to Semantic Scholar...
Your API Key has been approved! Beginning Search...

Success! Found 2567950 total papers.

Title: HOW COULD HUSSERL’S THEORY OF THE BODILY SELF-CONSTITUTION OF THE EGO HELP BRIDGE THE EXPLANATORY GAP?
Year: 2024
Abstract: The explanatory gap—the apparently ineliminable chasm between physical, bodily processes and states ...

Title: Moral Uncertainty in Technomoral Change: Bridging the Explanatory Gap
Year: 2022
Abstract: Abstract This paper explores the role of moral uncertainty in explaining the morally disruptive char...

Title: Neuro-cognitive multilevel causal modeling: A framework that bridges the explanatory gap between neuronal activity and cognition
Year: 2023
Abstract: Explaining how neuronal activity gives rise to cognition arguably remains the most significant chall...

Title: Electromagnetism’s Bridge Across the Explanatory Gap: How a Neuroscience/Physics Collaboration Delivers Explanation Into All 